# Meta-labeling и дополнительные target-кандидаты

**Шаг 3 экспериментов.** Meta-labeling: первичный сигнал (rd_regime) + вторичная разметка (triple-barrier) для фильтрации сделок. Альтернативные target-кандидаты.

## 1. Идея meta-labeling

- Первичный сигнал: rd_regime (BUY/SELL по знаку rd_value.diff).
- Вторичный: triple-barrier подтверждает или опровергает сделку.
- Target = 1 если rd_regime совпал с исходом TB (TP), -1 если не совпал (SL).

## 2. Импорты и загрузка

In [ ]:
import sys, os, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import joblib

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)
import lightgbm as lgb

labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')
if not os.path.exists(labeled_path):
    raise FileNotFoundError('Запустите 04_Data_Labeling_And_Feature_Loading.ipynb')
if not os.path.exists(feat_path):
    raise FileNotFoundError('Запустите 06_Feature_Selection.ipynb')

df = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    FEATURES = [l.strip() for l in f if l.strip()]
FEATURES = [c for c in FEATURES if c in df.columns]

h, mult = 5, 1.5

def add_tb(df_in, h=h, mult=mult):
    d = df_in.copy()
    labels = pd.Series(index=d.index, dtype=np.float64)
    for skey, g in d.groupby('session_key', sort=False):
        idx, c = g.index, g['close_price'].values
        h_arr, l_arr = g['high'].values, g['low'].values
        v = g['volatility_14'].fillna(0.01).clip(lower=1e-6).values
        lab = np.full(len(g), np.nan, dtype=np.float64)
        for i in range(len(g) - h):
            up, dn = c[i] * (1 + mult * v[i]), c[i] * (1 - mult * v[i])
            for j in range(1, h + 1):
                if h_arr[i + j] >= up: lab[i] = 1; break
                if l_arr[i + j] <= dn: lab[i] = -1; break
            else: lab[i] = 0
        labels.loc[idx] = lab
    d['tb'] = labels.values
    return d

df = add_tb(df)
print(f'Строк: {len(df):,}')

## 3. Meta-label: только когда rd_regime дал сигнал

meta_target = 1 если (rd_regime == tb), -1 если (rd_regime != tb). Берём только бары где rd_regime in {-1, 1}.

In [ ]:
df['rd_regime'] = df['rd_regime'].astype(float)
df['meta_target'] = np.where(df['rd_regime'] == df['tb'], 1.0, -1.0)
df.loc[~df['rd_regime'].isin([-1.0, 1.0]), 'meta_target'] = np.nan
df.loc[~df['tb'].isin([-1.0, 1.0]), 'meta_target'] = np.nan

vc = df['meta_target'].dropna().value_counts()
print('Meta-target распределение:', vc.to_dict())
print('Доля correct (1):', vc.get(1.0, 0) / vc.sum() if vc.sum() else 0)

## 4. Обучение и бектест: tb_vol vs meta_target

In [ ]:
def run_model(target_col, commission=0.001, seed=42):
    valid = df.dropna(subset=FEATURES + [target_col]).copy()
    valid = valid[valid[target_col].isin([-1.0, 1.0])]
    valid['y'] = (valid[target_col] == 1).astype(int)
    valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)
    
    sessions = valid['session_key'].unique().tolist()
    np.random.seed(seed)
    np.random.shuffle(sessions)
    train_s = set(sessions[:int(0.7*len(sessions))])
    test_s = set(sessions[int(0.85*len(sessions)):])
    train_df = valid[valid['session_key'].isin(train_s)]
    test_df = valid[valid['session_key'].isin(test_s)]
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[FEATURES].fillna(0))
    X_test = scaler.transform(test_df[FEATURES].fillna(0))
    model = lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=seed, verbose=-1)
    model.fit(X_train, train_df['y'].values)
    auc = roc_auc_score(test_df['y'].values, model.predict_proba(X_test)[:, 1])
    
    bt = test_df.dropna(subset=['ret_next'])
    X_bt = scaler.transform(bt[FEATURES].fillna(0))
    p = model.predict_proba(X_bt)[:, 1]
    ret = bt['ret_next'].values
    trade = np.where(p >= 0.6, 1, np.where(p <= 0.4, -1, 0))
    pnl = (trade * ret).sum() * 100 - (trade != 0).sum() * commission * 100
    return auc, pnl, (trade != 0).sum()

auc_tp, pnl_tp, nt_tp = run_model('target')
auc_tb, pnl_tb, nt_tb = run_model('tb')
auc_meta, pnl_meta, nt_meta = run_model('meta_target')
print('tp_sl_1_05: AUC=', f'{auc_tp:.4f}', 'net PnL=', f'{pnl_tp:.2f}%', 'trades=', nt_tp)
print('tb (vol):   AUC=', f'{auc_tb:.4f}', 'net PnL=', f'{pnl_tb:.2f}%', 'trades=', nt_tb)
print('meta_target:AUC=', f'{auc_meta:.4f}', 'net PnL=', f'{pnl_meta:.2f}%', 'trades=', nt_meta)

## Итог шага 13

- Meta-labeling: target = совпадение rd_regime с исходом TB (vol-based).
- Сравнение AUC и net PnL: tp_sl_1_05 vs tb (vol) vs meta_target.